# TRAIN patch — ERA5 from the new Climate Data Store

This notebook is **self-contained**: it carries the corrected code and the diff
inside itself, and applies everything to a freshly downloaded TRAIN installation.
No external file is needed.

Copy it wherever you like and run it top to bottom.

## What it fixes

TRAIN (Toolbox for Reducing Atmospheric InSAR Noise) was written for the
ERA‑Interim netCDF files of the old MARS archive. The ERA5 files downloaded from
the Climate Data Store after the 2024 migration have a different layout, and TRAIN
cannot read them.

| # | Problem | Symptom |
|---|---------|---------|
| 1 | The vertical coordinate is called `pressure_level`, not `level`; the time one `valid_time`, not `time` | `Undefined function or variable 'level'` |
| 2 | The files contain `expver`, of type `NC_STRING` | The `eval([varname '= double(netcdf.getVar(...))'])` loop fails on the cast |
| 3 | The variable order in the file has changed | `lats = netcdf.getVar(ncid,1)` reads the wrong array → **wrong lat/lon grid with no error at all** |
| 4 | `pressure_level` is in decreasing order (1000→1 hPa), the old `level` was increasing | The unconditional `flipud` inverts the ordering with respect to the code's intent |
| 5 | Number of levels hard-coded to 37 | Wrong grids with a different level set |
| 6 | `model_type='era5'` has neither the datapath branch nor the file-name generation | `error('weather model type not supported...')` |
| 7 | The 6-hourly ERA‑Interim time list is used for ERA5 | Step 2 looks for files that step 1 never downloads |
| 8 | The interactive DEM check calls `input()` | `matlab -batch` aborts: *"Support for user input is required"* |

## ⚠️ Do not use `ncrename` as a workaround

The most widespread remedy for problem 1 is renaming the dimensions in place:

```bash
ncrename -d pressure_level,level -v pressure_level,level file.nc   # DO NOT DO THIS
```

**On netCDF‑4 this silently blanks the coordinate variables.** The `z`/`t`/`r`
payload stays intact, but `level` and `time` revert entirely to the fill value
(NaN). The result is an all-NaN delay map, with no errors.

The patch already reads both spellings, so `ncrename` is unnecessary. On top of
that it **recovers files that are already damaged**: it rebuilds the levels from
an intact sibling file or from the standard ECMWF set, and orients them by
comparing against the geopotential (which grows as pressure decreases), so the
level↔index pairing is verified against the data rather than assumed.

The last section of the notebook contains a **read-only diagnostic** to find out
whether your files have been damaged.

---
## 1. Configuration

Tell it where TRAIN is. The notebook tries to find it on its own; if it gets it
wrong, set `TRAIN_DIR` by hand.

In [ ]:
import os, sys, shutil, subprocess, datetime, glob, re

# --- set the path here if autodetection fails -------------------------------
TRAIN_DIR = None          # e.g. "/home/user/software/TRAIN"
# ---------------------------------------------------------------------------

def _looks_like_train(p):
    return bool(p) and os.path.isfile(os.path.join(p, "matlab", "aps_load_era.m"))

if not _looks_like_train(TRAIN_DIR):
    candidates = [
        os.environ.get("APS_toolbox"),
        os.path.expanduser("~/software/TRAIN"),
        os.path.expanduser("~/TRAIN"),
        os.path.expanduser("~/Documents/software/TRAIN"),
        os.path.abspath("."),
        os.path.abspath(".."),
    ]
    candidates += sorted(glob.glob(os.path.expanduser("~/**/TRAIN"), recursive=True))[:5]
    for c in candidates:
        if _looks_like_train(c):
            TRAIN_DIR = os.path.abspath(c)
            break

if not _looks_like_train(TRAIN_DIR):
    raise SystemExit(
        "TRAIN not found. Set TRAIN_DIR by hand in the cell above.\n"
        "It must be the folder that contains matlab/aps_load_era.m"
    )

MATLAB_DIR = os.path.join(TRAIN_DIR, "matlab")
PATCH_DIR  = os.path.join(TRAIN_DIR, "patches")
os.makedirs(PATCH_DIR, exist_ok=True)
print("TRAIN_DIR :", TRAIN_DIR)
print("patches   :", PATCH_DIR)

## 2. Preliminary checks

Verifies that the required tools are there and records the starting state.
`patch` is indispensable; `git` and `matlab` are optional (they are only used for
the checks).

In [ ]:
def have(tool):
    return shutil.which(tool) is not None

print("patch  :", "OK" if have("patch") else "MISSING   <-- required")
print("git    :", "OK" if have("git") else "absent (reduced checks)")
print("matlab :", "OK" if have("matlab") else "absent (skipping the syntax check)")
print("ncdump :", "OK" if have("ncdump") else "absent (skipping the data diagnostic)")

if not have("patch"):
    raise SystemExit("Install GNU patch:  sudo apt install patch")

TARGETS = [
    "matlab/aps_load_era.m",
    "matlab/get_DEM.m",
    "matlab/get_gmt_version.m",
    "matlab/aps_weather_model_SAR.m",
    "matlab/aps_weather_model_InSAR.m",
    "matlab/aps_weather_model_filenames.m",
]

missing = [f for f in TARGETS if not os.path.isfile(os.path.join(TRAIN_DIR, f))]
if missing:
    raise SystemExit("Missing TRAIN files: %s" % missing)
print("\nall %d files to be patched are present" % len(TARGETS))

IS_GIT = os.path.isdir(os.path.join(TRAIN_DIR, ".git"))
if IS_GIT:
    out = subprocess.run(["git", "-C", TRAIN_DIR, "status", "--short"] + TARGETS,
                         capture_output=True, text=True).stdout.strip()
    print("\nlocal changes already present:" if out else "\nworking tree clean on the files to be patched")
    print(out)

## 3. Backup

Every file is copied into `patches/backup_<timestamp>/` before being touched. The
last cell of the notebook knows how to restore them.

In [ ]:
STAMP = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
BACKUP_DIR = os.path.join(PATCH_DIR, "backup_" + STAMP)
os.makedirs(BACKUP_DIR, exist_ok=True)

for f in TARGETS:
    dst = os.path.join(BACKUP_DIR, f.replace("/", "__"))
    shutil.copy2(os.path.join(TRAIN_DIR, f), dst)
    print("saved", os.path.basename(dst))

print("\nbackup in:", BACKUP_DIR)

## 4. Payload A — `matlab/aps_load_era.m`

This file is **rewritten in full**: the changes are too extensive for a positional
diff to be reliable across different versions of TRAIN. The content below is the
complete final file.

In [ ]:
APS_LOAD_ERA = r"""function [ Temp,WVapour,Geopot,Pressure,longrid,latgrid,xx,yy,lon0360_flag] = aps_load_era(file,era_data_type)
% loading ERA-I or ERA5 data from ECMWF website or ERA-I from BADC website
% Bekaert David
% modifications
% DB    10/04/2016  extract code from aps_era_SAR.m to make code modular
% DB 	07/06/2017  Update syntax to include ERA5 model
% --    2026        Compatibility with the "new" Climate Data Store (CDS-Beta /
%                   ecmwf-datastores) ERA5 netCDF files, see NEW-CDS notes below.

%%% Example on how to load netcdf files
% ncid = netcdf.open(file,'NC_NOWRITE');
% [numdims,numvars,numglobalatts,unlimdimid] = netcdf.inq(ncid);
% [dimname, dimlen] = netcdf.inqDim(ncid,0);
%
%         for k=1:numvars
%             [dimname, dimlen] = netcdf.inqVar(ncid,k-1);
%             fprintf([num2str(k-1) ' - ' dimname '\n'])
%         end

% NEW-CDS: files retrieved from the CDS after the 2024 migration differ from
% the legacy MARS/ERA-I netCDF this function was written for:
%   1. the vertical coordinate is called "pressure_level" instead of "level"
%      and the time coordinate "valid_time" instead of "time";
%   2. they carry an "expver" variable of netCDF type NC_STRING, which cannot
%      be cast with double() and used to abort the variable-reading loop;
%   3. the variables are not stored in the legacy order, so reading latitude
%      and longitude by their numeric id (1 and 0) returns the wrong arrays;
%   4. "pressure_level" is stored in decreasing order (1000 -> 1 hPa) whereas
%      the legacy "level" was increasing (1 -> 1000 hPa), so an unconditional
%      flip no longer produces the surface-first ordering TRAIN expects;
%   5. an in-place `ncrename` of the pressure_level/valid_time dimensions (a
%      common workaround) silently blanks those coordinate variables in
%      netCDF-4 files: the payload (z/t/r) survives but the pressure levels
%      come back as all-NaN. Such files are recovered below instead of
%      producing an all-NaN delay map.

% debug figure to test and validate dataloading.
debug_fig = 0;

% open the netcdf
ncid = netcdf.open(file,'NC_NOWRITE');

% read netcdf variables and get number of variables
[numdims,numvars,numglobalatts,unlimdimid] = netcdf.inq(ncid);


%% Swapping between BADC and ECMWF website data
if strcmpi(era_data_type,'ECMWF')
    % ECMWF data has field data and a scale plus offset.
    % Depending if this exist its added to the data.
    % NEW-CDS: collected in a struct keyed by variable name rather than in
    % eval-created workspace variables, so that variables can be looked up by
    % name and non-numeric ones (expver) can be skipped.
    ncvars = struct();
    for i = 0:numvars-1
        [varname, xtype, dimids, numatts] = netcdf.inqVar(ncid,i);
        scale = [];
        offset = [];
        for j = 0:numatts - 1
            attname1 = netcdf.inqAttName(ncid,i,j);

            if strcmp('add_offset',attname1)
                offset = netcdf.getAtt(ncid,i,attname1);
            end

            if strcmp('scale_factor',attname1)
                scale = netcdf.getAtt(ncid,i,attname1);
            end
        end

        raw = netcdf.getVar(ncid,i);
        % NEW-CDS: skip anything that is not a numeric field (NC_CHAR /
        % NC_STRING, e.g. "expver"), double() would error out on those.
        if (isnumeric(raw) || islogical(raw)) && isvarname(varname)
            data = double(raw);
            if ~isempty(scale)
                data = data*double(scale);
                if ~isempty(offset)
                    data = data + double(offset);
                end
            end
            ncvars.(varname) = data;
            clear data
        end
        clear varname xtype dimids numatts scale offset raw
    end

    % NEW-CDS: fetch the fields by name, accepting both the legacy and the
    % current CDS spellings of the vertical coordinate.
    Temp    = aps_load_era_getvar(ncvars,{'t'});
    Hum     = aps_load_era_getvar(ncvars,{'r'});
    Geopot  = aps_load_era_getvar(ncvars,{'z'});
    Plevs   = aps_load_era_getvar(ncvars,{'level','pressure_level','isobaricInhPa'});

    if isempty(Temp) || isempty(Hum) || isempty(Geopot)
        netcdf.close(ncid)
        error(['aps_load_era: t, r or z missing from ' file])
    end

    n_levels = size(Temp,3);

    % NEW-CDS: recover the pressure levels when the coordinate variable is
    % absent or has been blanked by an in-place ncrename.
    if isempty(Plevs) || numel(Plevs)~=n_levels || ~all(isfinite(Plevs))
        Plevs = aps_load_era_recover_plevs(file,n_levels,Geopot);
    end
    Plevs = Plevs(:);

    % TRAIN expects the vertical axis surface-first (highest pressure at index
    % 1). The legacy code achieved this with an unconditional flip of an
    % increasing "level"; sorting explicitly gives the identical result for
    % legacy files and also handles the decreasing "pressure_level" of the new
    % CDS files.
    [Plevs,ix_lev] = sort(Plevs,'descend');
    Temp   = Temp(:,:,ix_lev);
    Hum    = Hum(:,:,ix_lev);
    Geopot = Geopot(:,:,ix_lev);

    % NEW-CDS: latitude/longitude read by name, the variable order in the file
    % is no longer the legacy one.
    lats = aps_load_era_getvar(ncvars,{'latitude','lat'});
    lons = aps_load_era_getvar(ncvars,{'longitude','lon'});

elseif strcmpi(era_data_type,'BADC')
    % This is for ERA-I from BADC
    % Datafield are structured differently with different names
    % than ECMWF website data.

    % variables at each node 20 (0-19)
    [varname, vartype, dimids, natts] = netcdf.inqVar(ncid,0);

    % load data for variables 5 (temp), 11 (relative humidity) and 4 (geopotential height)
    Temp = double(netcdf.getVar(ncid,5));       % temp in K
    Hum = double(netcdf.getVar(ncid,11));       % relative humidity in percent
    Geopot = double(netcdf.getVar(ncid,4));     % Geopotential in m^2/s^2
    Plevs = double(netcdf.getVar(ncid,2));      % 37 pressure levels in hPa (or millibars) 1000 hPa at surface
    lats = [];
    lons = [];
end

% Permute to a 3 D grid
Temp = permute(Temp,[2,1,3]);
Hum = permute(Hum,[2,1,3]);
Geopot = permute(Geopot,[2,1,3]);


% Same for lats and lons
% NEW-CDS: only fall back to the positional read when the named lookup above
% did not resolve (BADC files, or an unexpected layout).
if isempty(lats) || isempty(lons)
    lats = double(netcdf.getVar(ncid,1));
    lons = double(netcdf.getVar(ncid,0));
end
lats = double(lats(:));
lons = double(lons(:));
n_latitude_points = size(lats,1);
n_longitude_points = size(lons,1);


% close nedtcdf
netcdf.close(ncid)


% adapting to the right lon lat grid size
n_levels = numel(Plevs);
Pressure = repmat(Plevs,[1,n_latitude_points,n_longitude_points]);
Pressure = permute(Pressure,[2,3,1]);


% NEW-CDS: use the actual number of pressure levels instead of a hard-coded 37
latgrid = repmat(lats,[1,n_levels,n_longitude_points]);
latgrid = permute(latgrid,[1,3,2]);
longrid = repmat(lons,[1,n_levels,n_latitude_points]);
longrid = permute(longrid,[3,1,2]);

% Get list of points to look at in analysis
[xx,yy] = meshgrid(1:n_longitude_points,1:n_latitude_points);


% (see IFS documentation part 2: Data assimilation (CY25R1)).
% Calculate saturated water vapour pressure (svp) for water
% (svpw) using Buck 1881 and for ice (swpi) from Alduchow
% and Eskridge (1996) euation AERKi
svpw = 6.1121.*exp((17.502.*(Temp-273.16))./(240.97+Temp-273.16));
svpi = 6.1121.*exp((22.587.*(Temp-273.16))./(273.86+Temp-273.16));
tempbound1 = 273.16; %0
tempbound2 = 250.16; %-23
svp = svpw;

% Faster expression
wgt = (Temp - tempbound2)/(tempbound1 - tempbound2);
svp = svpi+ (svpw - svpi).*wgt.^2;
ix_bound1 = find(Temp > tempbound1);
svp(ix_bound1) = svpw(ix_bound1);
ix_bound2 = find(Temp < tempbound2);
svp(ix_bound2) = svpi(ix_bound2);
WVapour = Hum./100.*svp;
clear Hum

% inform about the organisation of the longitudes
if sum(lons>180)>1
    lon0360_flag = 'y';
else
    lon0360_flag = 'n';
end

% validation plots
if debug_fig ==1
    figure('position',[ 3         628        1402         586]);
    subplot(2,5,1)
    imagesc(Temp(:,:,end))
    colorbar
    axis xy
    axis equal
    axis tight
    title('temp upper atmo')
    subplot(2,5,2)
    imagesc(Temp(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('temp lower atmo')

    subplot(2,5,3)
    imagesc(Pressure(:,:,end))
    colorbar
    axis xy
    axis equal
    axis tight
    title('pressure upper atmo')
    subplot(2,5,4)
    imagesc(Pressure(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('pressure lower atmo')



    subplot(2,5,6)
    imagesc(WVapour(:,:,end))
    colorbar
    axis xy
    axis equal
    axis tight
    title('Water vapour upper atmo')
    subplot(2,5,7)
    imagesc(WVapour(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('Water vapour lower atmo')


    subplot(2,5,8)
    imagesc(Geopot(:,:,end))
    colorbar
    axis xy
    axis equal
    axis tight
    title('geopotential upper atmo')
    subplot(2,5,9)
    imagesc(Geopot(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('geopotential lower atmo')


    subplot(2,5,5)
    imagesc(latgrid(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('lat')
    subplot(2,5,10)
    imagesc(longrid(:,:,1))
    colorbar
    axis xy
    axis equal
    axis tight
    title('lon')
end



end


function data = aps_load_era_getvar(ncvars,names)
% NEW-CDS: return the first field of the ncvars struct matching one of names,
% or [] when none of them is present.
data = [];
for k = 1:numel(names)
    if isfield(ncvars,names{k})
        data = ncvars.(names{k});
        return
    end
end
end


function Plevs = aps_load_era_recover_plevs(file,n_levels,Geopot)
% NEW-CDS: rebuild the pressure level vector of a weather model file whose
% vertical coordinate variable is missing or has been blanked (all NaN) by an
% in-place `ncrename` on a netCDF-4 file. The z/t/r payload of such files is
% intact, only the coordinate is lost, so the run can be salvaged without
% touching the data on disk.
%
% Sources, in order of preference:
%   1. a sibling weather model file of the same run that still has a readable
%      vertical coordinate;
%   2. the standard ECMWF pressure level sets (37 or 25 levels).
% The recovered vector is then oriented against the geopotential so that it is
% paired with the correct level index regardless of how the file stores them.

Plevs = [];
level_names = {'level','pressure_level','isobaricInhPa'};

% ---- 1. sibling files ---------------------------------------------------
[fdir,~,fext] = fileparts(file);
search_dirs = {fdir};
parentdir = fileparts(fdir);
if ~isempty(parentdir)
    sub = dir(parentdir);
    for k = 1:numel(sub)
        if sub(k).isdir && ~strcmp(sub(k).name,'.') && ~strcmp(sub(k).name,'..')
            search_dirs{end+1} = fullfile(parentdir,sub(k).name); %#ok<AGROW>
        end
    end
end

for s = 1:numel(search_dirs)
    cand = dir(fullfile(search_dirs{s},['*' fext]));
    for c = 1:numel(cand)
        candfile = fullfile(search_dirs{s},cand(c).name);
        if strcmp(candfile,file)
            continue
        end
        try
            info = ncinfo(candfile);
        catch
            continue
        end
        vn = {info.Variables.Name};
        for L = 1:numel(level_names)
            if any(strcmp(vn,level_names{L}))
                try
                    trial = double(ncread(candfile,level_names{L}));
                catch
                    continue
                end
                trial = trial(:);
                if numel(trial)==n_levels && all(isfinite(trial)) && all(trial>0)
                    Plevs = trial;
                    fprintf(['aps_load_era: pressure levels of ' file ...
                             ' are unreadable, recovered from ' candfile '\n']);
                    break
                end
            end
        end
        if ~isempty(Plevs), break, end
    end
    if ~isempty(Plevs), break, end
end

% ---- 2. standard ECMWF pressure level sets ------------------------------
if isempty(Plevs)
    plevs37 = [1000 975 950 925 900 875 850 825 800 775 750 700 650 600 550 ...
               500 450 400 350 300 250 225 200 175 150 125 100 70 50 30 20 ...
               10 7 5 3 2 1]';
    plevs25 = [1000 975 950 925 900 875 850 825 800 775 750 700 650 600 550 ...
               500 450 400 350 300 250 225 200 175 150]';
    if n_levels==numel(plevs37)
        Plevs = plevs37;
    elseif n_levels==numel(plevs25)
        Plevs = plevs25;
    else
        error(['aps_load_era: the pressure levels of ' file ' are unreadable ' ...
               '(likely blanked by an in-place ncrename) and cannot be ' ...
               'reconstructed for ' num2str(n_levels) ' levels. Re-download ' ...
               'this file from the CDS.']);
    end
    fprintf(['aps_load_era: pressure levels of ' file ' are unreadable, ' ...
             'falling back on the standard ' num2str(n_levels) ...
             '-level ECMWF set\n']);
end

% ---- 3. orient against the geopotential ---------------------------------
% Geopotential grows with altitude, i.e. with decreasing pressure. Whichever
% end of the level axis carries the smaller geopotential is the surface end
% and must be paired with the highest pressure.
z_first = mean(reshape(Geopot(:,:,1),[],1),'omitnan');
z_last  = mean(reshape(Geopot(:,:,end),[],1),'omitnan');
if z_first <= z_last
    Plevs = sort(Plevs,'descend');
else
    Plevs = sort(Plevs,'ascend');
end
end
"""

print("characters:", len(APS_LOAD_ERA), "| lines:", APS_LOAD_ERA.count(chr(10)))
for marker in ["aps_load_era_recover_plevs", "pressure_level", "valid_time",
               "isnumeric(raw)", "sort(Plevs,'descend')"]:
    print("  %-32s %s" % (marker, "present" if marker in APS_LOAD_ERA else "MISSING"))

In [ ]:
dst = os.path.join(MATLAB_DIR, "aps_load_era.m")
with open(dst, "w") as fh:
    fh.write(APS_LOAD_ERA)
print("written:", dst)
print("bytes on disk:", os.path.getsize(dst))

## 5. Payload B — diff for the other four files

Small, surgical changes, applied with `patch -p1 --forward`:

- **`aps_weather_model_SAR.m`** — `era5` branch for the datapath, and the
  **hourly** list for ERA5, aligned with the one `aps_era5_files.m` actually uses
  to download;
- **`aps_weather_model_InSAR.m`** — `era5` branch for the datapath;
- **`aps_weather_model_filenames.m`** — `ggap*.nc` name generation for `era5`;
- **`get_DEM.m`** — reading `n_columns`/`n_rows` with `grdinfo -C` (GMT 6
  compatibility) and the interactive prompt skipped under `matlab -batch`;
- **`get_gmt_version.m`** — recognises the GMT 6 `usage:`.

`--forward` makes the cell **idempotent**: if the patch is already applied, the
hunks are skipped instead of being reversed.

> `APS_CONFIG.sh` is not included: it holds the path of your own installation,
> which has to be set by hand.

In [ ]:
OTHERS_DIFF = r"""diff --git a/matlab/aps_weather_model_InSAR.m b/matlab/aps_weather_model_InSAR.m
index 8113ae4..6c5b1d7 100644
--- a/matlab/aps_weather_model_InSAR.m
+++ b/matlab/aps_weather_model_InSAR.m
@@ -62,15 +62,17 @@ ll_matfile = getparm_aps('ll_matfile',1);
 ifgday_matfile = getparm_aps('ifgday_matfile');
 
 model_type = lower(model_type);
-if strcmpi(model_type,'era')
+% era5 shares era_datapath with era; without this branch
+% aps_weather_model('era5',3,...) aborts on the error below.
+if strcmpi(model_type,'era') || strcmpi(model_type,'era5')
     weather_model_datapath = getparm_aps('era_datapath',1);
 elseif strcmpi(model_type,'merra') || strcmpi(model_type,'merra2')
     weather_model_datapath = getparm_aps('merra_datapath',1);
 elseif strcmpi(model_type,'gacos')
     weather_model_datapath = getparm_aps('gacos_datapath',1);
 else
-    error('Not a supported model: either: ERA, MERRA, MERRA2, GACOS')
-end 
+    error('Not a supported model: either: ERA, ERA5, MERRA, MERRA2, GACOS')
+end
 lambda = getparm_aps('lambda',1)*100;                       % radar wavelength in cm
 datestructure = 'yyyymmdd';                               % assumed date structure for era
 inc_angle =  getparm_aps('look_angle',1);
diff --git a/matlab/aps_weather_model_SAR.m b/matlab/aps_weather_model_SAR.m
index c5a94e3..deebef2 100644
--- a/matlab/aps_weather_model_SAR.m
+++ b/matlab/aps_weather_model_SAR.m
@@ -106,6 +106,15 @@ stamps_processed = getparm_aps('stamps_processed',1);
 if strcmp(model_type,'narr')
     timelist_model = ['0000' ;'0300'; '0600' ; '0900'; '1200' ;'1500'; '1800' ;'2100'; '0000'];
     model_lag = 8*7;    % days
+elseif strcmpi(model_type,'era5')
+    % ERA5 is distributed hourly. The 6 hourly ERA-Interim list would make
+    % this step look for time stamps that aps_era5_files.m never orders or
+    % downloads, so use the same hourly list as that function.
+    timelist_model= ['0000' ; '0100' ; '0200' ; '0300' ; '0400' ; '0500' ; ...
+                     '0600' ; '0700' ; '0800' ; '0900' ; '1000' ; '1100' ; ...
+                     '1200' ; '1300' ; '1400' ; '1500' ; '1600' ; '1700' ; ...
+                     '1800' ; '1900' ; '2000' ; '2100' ; '2200' ; '2300' ; '0000'];
+    model_lag = 0; % ? check lags
 else
     timelist_model= ['0000' ; '0600' ; '1200' ; '1800' ; '0000'];       % the time interval the model is outputed
     model_lag = 0; % ? check lags
@@ -114,16 +123,18 @@ era_data_type = [];                                                 % the weathe
 
 
 %%% Updating specific weather model information
-if strcmpi(model_type,'era')
+% era5 shares era_datapath and the ECMWF/BADC data type with era. Without
+% this branch aps_weather_model('era5',2,...) aborts on the error below.
+if strcmpi(model_type,'era') || strcmpi(model_type,'era5')
     weather_model_datapath = getparm_aps('era_datapath',1);
     era_data_type = getparm_aps('era_data_type');         % the datatype of the model either BADC or ERA
 elseif strcmpi(model_type,'merra') || strcmpi(model_type,'merra2')
-    weather_model_datapath = getparm_aps('merra_datapath',1); 
-elseif strcmpi(model_type,'narr') 
-    weather_model_datapath = getparm_aps('narr_datapath',1); 
+    weather_model_datapath = getparm_aps('merra_datapath',1);
+elseif strcmpi(model_type,'narr')
+    weather_model_datapath = getparm_aps('narr_datapath',1);
 
 else
-    error(['weather model type not supported, either: wrf, era, narr, merra for now'])
+    error(['weather model type not supported, either: wrf, era, era5, narr, merra for now'])
 end
 
 lambda = getparm_aps('lambda',1)*100;                       % radar wavelength in cm
diff --git a/matlab/aps_weather_model_filenames.m b/matlab/aps_weather_model_filenames.m
index f6512a7..690fbd8 100644
--- a/matlab/aps_weather_model_filenames.m
+++ b/matlab/aps_weather_model_filenames.m
@@ -34,7 +34,9 @@ if nargin<6
 end
 
 for d =1:size(date_before,1)
-    if strcmpi(model_type,'era')    
+    if strcmpi(model_type,'era') || strcmpi(model_type,'era5')
+        % era5 uses the same ggap naming and folder layout as era; without
+        % this branch modelfile_before/after stay undefined for era5.
         %Format ggapYYYYMMDDHHMM.nc
         modelfile_before(d,:) = [weather_model_datapath filesep date_before(d,:) filesep 'ggap' date_before(d,:) time_before(d,:) '.nc']; 
         modelfile_after(d,:) = [weather_model_datapath filesep date_after(d,:) filesep 'ggap' date_after(d,:) time_after(d,:) '.nc'];
diff --git a/matlab/get_DEM.m b/matlab/get_DEM.m
index ca4c60a..b15695f 100644
--- a/matlab/get_DEM.m
+++ b/matlab/get_DEM.m
@@ -364,27 +364,17 @@ dem = data(:,3);
 clear data data_vector
 
 
-%% load the resampled DEM
-if strcmp(gmt5_above,'y')
-    nncols_cmd = ['echo `' GMT_string 'grdinfo tmp_smp.grd | grep n_columns | awk ''{print $NF}''`>', path_dem ,filesep ,'temp3'];
-    nnrows_cmd = ['echo `' GMT_string 'grdinfo tmp_smp.grd | grep n_rows | awk ''{print $NF}''`>>', path_dem ,filesep ,'temp3'];
-else
-    nncols_cmd = ['echo `' GMT_string 'grdinfo tmp_smp.grd | grep nx | awk ''{print $NF}''`>', path_dem ,filesep ,'temp3'];
-    nnrows_cmd = ['echo `' GMT_string 'grdinfo tmp_smp.grd | grep ny | awk ''{print $NF}''`>>', path_dem ,filesep ,'temp3'];
-end
-aps_systemcall(nncols_cmd);
-aps_systemcall(nnrows_cmd);
-
-DEM_info = load([path_dem ,filesep 'temp3']);
-aps_systemcall(['rm ' path_dem ,filesep 'temp3']);
+%% load the resampled DEM  (PATCH: read grdinfo directly, no temp3 file)
+[~, ncol_str] = system([GMT_string 'grdinfo -C tmp_smp.grd | awk ''{print $10}''']);
+[~, nrow_str] = system([GMT_string 'grdinfo -C tmp_smp.grd | awk ''{print $11}''']);
+DEM_info = [str2double(ncol_str); str2double(nrow_str)];
 nncols = DEM_info(1);
 nnrows = DEM_info(2);
-clear DEM_info
-dem =reshape(dem,nncols,nnrows)';
+dem = reshape(dem,nncols,nnrows)';
 
 if fig_test ==1
     figure('name','DEM debug test');
-    imagesc([xmin xmax],[ymax ymin],dem)   
+    imagesc([xmin xmax],[ymax ymin],dem)
     colorbar
     view(0,90)
     axis equal
@@ -392,12 +382,20 @@ if fig_test ==1
     axis xy
 
     % check if this is correct
-    str='';
-    while ~strcmpi(str,'y') && ~strcmpi(str,'n')
-        fprintf(['Does the DEM look reasonable? \n'])
-        str = input('Continue? [y: for yes, n: no] \n','s');
-    end
-    if strcmpi(str,'n')
-        error('Check the dem input file.')
+    % HEADLESS: input() throws "Support for user input is required, which is
+    % not available on this platform" under `matlab -batch` / -nodisplay, which
+    % aborts any non-interactive run of the APS chain. Only prompt when there
+    % actually is somebody to answer.
+    if batchStartupOptionUsed
+        fprintf('Batch mode: skipping the interactive DEM check\n')
+    else
+        str='';
+        while ~strcmpi(str,'y') && ~strcmpi(str,'n')
+            fprintf(['Does the DEM look reasonable? \n'])
+            str = input('Continue? [y: for yes, n: no] \n','s');
+        end
+        if strcmpi(str,'n')
+            error('Check the dem input file.')
+        end
     end
 end
diff --git a/matlab/get_gmt_version.m b/matlab/get_gmt_version.m
index 42bdefb..5747290 100644
--- a/matlab/get_gmt_version.m
+++ b/matlab/get_gmt_version.m
@@ -140,7 +140,7 @@ end
 if gmt_function_does_not_work~=0
     [gmt_function_does_not_work, b] = system([GMT_string ' ' command_str]);
     if ~isempty(b)
-       ix = findstr('usage: psxy',b);
+       ix = findstr('usage:',b);
        if ~isempty(ix)
            gmt_function_does_not_work = 0;
            fprintf(['WARNING: GMT functions need to be preceded by ' GMT_string(1:end-1) ': e.g. ' GMT_string 'xyz2grd \n'])
"""

DIFF_PATH = os.path.join(PATCH_DIR, "TRAIN_ERA5_others.diff")
with open(DIFF_PATH, "w") as fh:
    fh.write(OTHERS_DIFF)
print("diff written to:", DIFF_PATH)
print("files touched:")
for line in OTHERS_DIFF.splitlines():
    if line.startswith("+++ b/"):
        print("   ", line[6:])

In [ ]:
def run_patch(*extra):
    return subprocess.run(
        ["patch", "-p1", "--forward", "--batch", "-i", DIFF_PATH] + list(extra),
        cwd=TRAIN_DIR, capture_output=True, text=True)

# dry run: says what would happen without writing anything
dry = run_patch("--dry-run")
print("--- dry run ---")
print(dry.stdout or dry.stderr)

already = "Reversed (or previously applied) patch detected" in (dry.stdout + dry.stderr)
if already:
    print(">>> the patch looks already applied (fully or partly): writing nothing")
elif dry.returncode != 0:
    print(">>> the dry run failed: your version of TRAIN differs too much.")
    print(">>> retrying with whitespace tolerance (-l) and maximum fuzz")
    dry = run_patch("--dry-run", "-l", "-F3")
    print(dry.stdout or dry.stderr)

if not already and dry.returncode == 0:
    res = run_patch("-l", "-F3")
    print("--- applying ---")
    print(res.stdout or res.stderr)
    print("exit code:", res.returncode)
elif not already:
    raise SystemExit(
        "patch not applicable. Apply the changes by hand following the diff "
        "printed by the previous cell, or restore from the backup."
    )

## 6. Verification

Searches the files for the textual signatures the patch must have introduced. If
one is missing, that part was not applied.

In [ ]:
CHECKS = {
    "matlab/aps_load_era.m": [
        ("coordinate lookup by name", "pressure_level"),
        ("NC_STRING variables skipped", "isnumeric(raw)"),
        ("explicit level ordering", "sort(Plevs,'descend')"),
        ("damaged-file recovery", "aps_load_era_recover_plevs"),
        ("no hard-coded 37", "n_levels,n_longitude_points"),
    ],
    "matlab/aps_weather_model_SAR.m": [
        ("era5 branch for the datapath", "strcmpi(model_type,'era') || strcmpi(model_type,'era5')"),
        ("hourly list for era5", "'0500' ;"),
    ],
    "matlab/aps_weather_model_InSAR.m": [
        ("era5 branch for the datapath", "strcmpi(model_type,'era') || strcmpi(model_type,'era5')"),
    ],
    "matlab/aps_weather_model_filenames.m": [
        ("file names for era5", "strcmpi(model_type,'era') || strcmpi(model_type,'era5')"),
    ],
    "matlab/get_DEM.m": [
        ("grdinfo -C for GMT 6", "grdinfo -C tmp_smp.grd"),
        ("prompt skipped in batch mode", "batchStartupOptionUsed"),
    ],
    "matlab/get_gmt_version.m": [
        ("GMT 6 usage: string", "findstr('usage:',b)"),
    ],
}

ok = True
for f, checks in CHECKS.items():
    text = open(os.path.join(TRAIN_DIR, f)).read()
    print(f)
    for label, needle in checks:
        found = needle in text
        ok &= found
        print("   [%s] %s" % ("x" if found else " ", label))

print()
print("ALL CHECKS PASSED" if ok else "WARNING: some change does not appear to be applied")

if IS_GIT:
    print("\n--- git diff --stat ---")
    print(subprocess.run(["git", "-C", TRAIN_DIR, "diff", "--stat"] + TARGETS,
                         capture_output=True, text=True).stdout)

## 7. MATLAB syntax check (optional)

If `matlab` is on the PATH, runs `checkcode` on the modified files. Skips without
error if MATLAB is not there.

In [ ]:
if not have("matlab"):
    print("matlab not found: skipping")
else:
    files_ml = ",".join("'%s'" % os.path.join(TRAIN_DIR, f) for f in TARGETS)
    script = (
        "files={%s};"
        "bad=0;"
        "for k=1:numel(files),"
        "  r=checkcode(files{k});"
        "  n=0;"
        "  for j=1:numel(r),"
        "    m=lower(r(j).message);"
        "    if ~isempty(strfind(m,'parse'))||~isempty(strfind(m,'unbalanc'))||~isempty(strfind(m,'invalid')),"
        "      fprintf('ERROR %%s L%%d: %%s\\n',files{k},r(j).line,r(j).message); n=n+1;"
        "    end,"
        "  end,"
        "  fprintf('%%-40s %%d messages, %%d errors\\n',files{k},numel(r),n);"
        "  bad=bad+n;"
        "end,"
        "fprintf('\\ntotal syntax errors: %%d\\n',bad);"
        "exit"
    ) % files_ml
    res = subprocess.run(["matlab", "-batch", script], capture_output=True, text=True)
    print(res.stdout or res.stderr)

---
## 8. ERA5 data diagnostic (read-only)

Checks whether your netCDF files have readable coordinates. **It changes
nothing.**

Set `ERA_DIR` to the folder given by `getparm_aps('era_datapath')`: the one with a
subfolder per date (`20251003/`, `20251015/`, …) containing the
`ggapYYYYMMDDHHMM.nc` files.

If any file shows up as `DAMAGED`, there is nothing to redo: the patch recovers
them on its own at read time, printing a warning line for each. It is still useful
to know which ones they are.

In [ ]:
ERA_DIR = ""     # <-- e.g. "/home/user/.../INSAR_master_data_XXX/ERA_5"

if not ERA_DIR:
    print("ERA_DIR not set: skipping the diagnostic")
elif not have("ncdump"):
    print("ncdump not available (sudo apt install netcdf-bin): skipping")
elif not os.path.isdir(ERA_DIR):
    print("folder does not exist:", ERA_DIR)
else:
    def read_levels(path):
        """(variable_name, text_of_the_values) or (None, None).

        The name has to be found by trying to read the variable, not with a
        substring match on the header: in files put through ncrename the
        `history` attribute still contains the word 'pressure_level' even
        though the variable is now called 'level'.
        """
        for cand in ("level", "pressure_level", "isobaricInhPa"):
            p = subprocess.run(["ncdump", "-v", cand, path], capture_output=True, text=True)
            if p.returncode != 0:
                continue
            parts = p.stdout.split("\ndata:", 1)
            if len(parts) < 2:
                continue
            m = re.search(r"(?m)^\s*" + re.escape(cand) + r"\s*=\s*([^;]*);", parts[1], re.S)
            if m:
                return cand, m.group(1)
        return None, None

    files = sorted(glob.glob(os.path.join(ERA_DIR, "*", "*.nc")))
    print("files found:", len(files), "\n")
    corrupt, fine, unknown = [], [], []
    for f in files:
        lev, vals = read_levels(f)
        if lev is None or not vals.strip():
            unknown.append(f)
            continue
        tokens = set(vals.replace(",", " ").split())
        # a coordinate blanked by ncrename comes back entirely as the fill value "_"
        if tokens <= {"_"}:
            corrupt.append((f, lev))
        else:
            fine.append((f, lev, vals.split(",")[0].strip()))

    print("INTACT     :", len(fine))
    print("DAMAGED    :", len(corrupt))
    print("UNREADABLE :", len(unknown))
    if fine:
        f0, lev0, v0 = fine[0]
        print("\nintact example : %s  (variable '%s', first level %s hPa)"
              % (os.path.relpath(f0, ERA_DIR), lev0, v0))
    if corrupt:
        print()
    for f, lev in corrupt[:40]:
        print("   DAMAGED", os.path.relpath(f, ERA_DIR), "(variable '%s' all NaN)" % lev)
    if len(corrupt) > 40:
        print("   ... and", len(corrupt) - 40, "more")
    for f in unknown[:10]:
        print("   UNREADABLE", os.path.relpath(f, ERA_DIR))
    if corrupt and fine:
        print("\nThere are intact files in the same tree: the patch will rebuild the levels from those.")
    elif corrupt and not fine:
        print("\nNo intact file to copy from: the patch will use the standard 37-level ECMWF set.")

---
## 9. How to run the APS correction

With the patch applied both routes work. For a satellite pass at 05:10 UTC:

| route | files needed | interpolation weights | field in `tca2.mat` |
|-------|--------------|-----------------------|---------------------|
| `aps_weather_model('era',2,4)`  | `ggap…0000.nc` + `ggap…0600.nc` | 0.139 / 0.861 | `ph_tropo_era` |
| `aps_weather_model('era5',2,4)` | `ggap…0500.nc` + `ggap…0600.nc` | 0.833 / 0.167 | `ph_tropo_era5` |

```matlab
addpath(genpath('/path/to/StaMPS/matlab'))
addpath(genpath('/path/to/TRAIN/matlab'))
cd /path/to/project

setparm_aps('era_datapath', '/absolute/path/to/ERA_5')
setparm_aps('era_data_type', 'ECMWF')
setparm_aps('UTC_sat', '05:10')
setparm_aps('stamps_processed', 'y')

aps_weather_model('era5', 0, 0)   % dry run: shows the dates and expected files
aps_weather_model('era5', 1, 1)   % download from the CDS (needs ~/.cdsapirc)
aps_weather_model('era5', 2, 4)   % compute the delays and write tca2.mat
```

Then, in StaMPS:

```matlab
ps_plot('a','a_era5')          % estimated APS
setparm('subtr_tropo','y')
setparm('tropo_method','a_era5')
ps_plot('v-dao','a_era5')      % corrected velocity
```

Replace `a_era5` with `a_erai` to use the field from the `era` route.

### On choosing between the two routes

The `era5` route samples much closer to the acquisition, so in theory it is
preferable. On a test dataset (23 Sentinel‑1 images, ~40 km, elevation range
247–1898 m) the two corrections turned out to be **equivalent**, correlated at
0.983 with each other: the residual phase–topography correlation drops from 0.551
to 0.343 (`era`) and 0.339 (`era5`). Look at both fields before deciding which one
to put in `tropo_method` — over a different area the verdict may change.

---
## 10. Rollback

Puts the files back as they were before this notebook was run. Change
`DO_ROLLBACK` to `True` to run it.

In [ ]:
DO_ROLLBACK = False

if not DO_ROLLBACK:
    print("rollback disabled. Backup available in:", BACKUP_DIR)
else:
    for f in TARGETS:
        src = os.path.join(BACKUP_DIR, f.replace("/", "__"))
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(TRAIN_DIR, f))
            print("restored", f)
    print("\ndone.")